In [1]:
import os
from datetime import datetime

import pandas as pd
from tqdm import tqdm

import wandb

In [2]:
%cd "/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/"

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [3]:
# Konfiguration
CSV_PATH = "test_grid_experiments_epg.csv"
WANDB_ENTITY = "roesch01-university-of-mannheim"  # Falls nötig, hier deinen Usernamen/Teamnamen eintragen
WANDB_PROJECT = "epg"  # Falls nötig, hier den Projektnamen eintragen
TAG_FILTER = "result"

date_threshold = datetime(year=2026, month=2, day=5)

In [4]:
def update_csv_with_wandb_ids():
    # 1. CSV laden
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"Die Datei {CSV_PATH} wurde nicht gefunden.")
    
    df = pd.read_csv(CSV_PATH, sep=';')
    
    # Spalte für ID vorbereiten, falls noch nicht da
    df['wandb_run_id'] = None

    # 2. W&B API initialisieren
    api = wandb.Api()

    print(f"Starte Abgleich für {len(df)} Zeilen...")

    # 3. Iteration mit Fortschrittsbalken
    pbar = tqdm(df.iterrows(), total=len(df), desc="Matching Runs")
    for index, row in pbar:

        # Filter-Kriterien aus der CSV Zeile
        # Hinweis: Wir casten lambda_epg auf float, falls es als String geladen wurde
        dataset = row['dataset']
        attributor = row['attributor']
        lambda_epg = row['lambda_epg']
        epg_lvl = row['epg_lvl']

        pbar.set_postfix({"dataset": dataset, "attributor":attributor, "lambda_epg":lambda_epg, "epg_lvl":epg_lvl})
        
        # W&B Query (MongoDB Syntax)
        # Wir filtern nach Projekt, Tag und den drei spezifischen Config-Parametern
        filters = {
            "tags": {"$in": [TAG_FILTER]},
            "config.dataset": dataset,
            "config.attributor_name": attributor,
            "config.hyperparameters.lambda_epg": lambda_epg,
            "config.hyperparameters.epg_lvl": epg_lvl,
            "created_at": {"$gt": date_threshold.isoformat()},
            "state": {"$ne": "running"}
        }
        
        # if not pd.isna(top_k_percent):
        #     filters["config.hyperparameters.top_k_percent"] = top_k_percent
        # else:
        #     filters["config.hyperparameters.top_k_percent"] = None

        
        runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_ENTITY else WANDB_PROJECT, filters=filters)
        
        
        try:
            run_list = list(runs)

        except Exception as e:
            filters_readable = '\n'.join([f'{k}: {v}' for k, v in filters.items()])
            print(f"FEHLER bei W&B API Anfrage für Zeile {index}: {e}")
            print(f"Filter: {filters_readable}")
            continue
            raise

        # 4. Validierung & Fehlerbehandlung
        if len(run_list) == 0:
            filters_readable = '\n'.join([f'{k}: {v}' for k, v in filters.items()])
            print(f"FEHLER: Kein Run gefunden für:\n"
                f"Filter: {filters_readable}")
            continue
            raise ValueError(
                f"FEHLER: Kein Run gefunden für:\n"
                f"Filter: {filters_readable}"
            )
        
        if len(run_list) > 1:
            run_ids = [r.id for r in run_list]
            filters_readable = '\n'.join([f'{k}: {v}' for k, v in filters.items()])
            print(filters_readable)
            continue
            raise ValueError(
                f"FEHLER: Mehrere Runs ({len(run_list)}) gefunden für:\n"
                f"Filter: {filters_readable}\n"
                f"IDs: {run_ids}"
            )

        run = run_list[0]

        if (df["wandb_run_id"] == run.id).any():
            raise ValueError(f"Run {run.id} existiert bereits im DataFrame.")
        
        # ID eintragen
        df.at[index, 'wandb_run_id'] = run_list[0].id

    return df

In [ ]:
data = update_csv_with_wandb_ids()

Starte Abgleich für 86 Zeilen...


Matching Runs:   0%|          | 0/86 [00:00<?, ?it/s, dataset=FunnyBirds, attributor=GradCamAttributor, lambda_epg=0.001, epg_lvl=concept]

Matching Runs:   0%|          | 0/86 [00:00<?, ?it/s, dataset=FunnyBirds, attributor=GradCamAttributor, lambda_epg=0.001, epg_lvl=concept]


KeyboardInterrupt: 

In [10]:
# 5. Speichern
data.to_csv(CSV_PATH, index=False, sep=';')
print(f"\nErfolg! Die CSV wurde aktualisiert und gespeichert unter: {CSV_PATH}")


Erfolg! Die CSV wurde aktualisiert und gespeichert unter: test_grid_experiments_epg.csv
